# Important Reminder {.unnumbered}

- This assignment must be completed using **Google Colab** and **Google Drive**.
- You must use the **same two rival companies and NAICS vertical** selected in Assignment 1.
- You will reuse the corpus built in Assignment 1.

:::{.callout-important}
## Prerequisites

The following files from Assignment 1 must exist on your Drive before starting:

- `/assignment01/corpus.csv`
- `/assignment01/chunks.csv`
- `/assignment01/vocab.json`

Load and inspect each file at the top of your notebook before proceeding. If any file is missing, complete Assignment 1 first.
:::

:::{.callout-important}
## DataFrame and Visualization Standard

For every DataFrame you create or transform:

- Show `.head()` to display sample rows in your notebook
- Run `.info()` to confirm column types and null counts
- Run `.describe()` to summarize numeric statistics
- Include at least one relevant visualization (bar chart, heatmap, scatter plot, or distribution plot) with a descriptive title and labeled axes

These are graded as part of your technical evidence. Undocumented DataFrames and unexplained outputs will lose points.
:::

# Submission Instructions {.unnumbered}

Submit three files on Blackboard:

1. **Word Report (`.docx`)** — maximum 10 pages, APA formatted
   - Answers to Q1–Q3 and Business Recommendation
   - Include a title page and references in APA style
   - Embed all required tables and figures inline and label them clearly
   - No code

2. **Jupyter Notebook (`.ipynb`)** — complete and fully executable on Colab, organized by section
   - Include all required code, outputs, tables, and charts
   - The notebook must run top-to-bottom in a fresh Colab runtime
   - Save all required intermediate artifacts to Drive

3. **AI Disclosure Document**
   - Submit as a separate file
   - Document all AI tools used, complete prompts, output excerpts, validation steps, and share links when available
   - This is graded as part of responsible AI use and reproducibility

:::{.callout-tip}
## What Strong Submissions Consistently Do Well

- Keep the report concise and executive-facing rather than turning it into a notebook transcript
- Make every number in the report match executed notebook output exactly
- Save intermediate artifacts cleanly so later questions reuse the same corpus, models, metrics, and visualizations
- Show diagnostic evidence and failure analysis, not only final accuracy
- Disclose AI use specifically: which tool, for which step, with what prompt and output, and how the result was verified
- Preserve enough prompt-and-output context that a grader can understand your reasoning process
:::

## Word Report Expectations

Use the report to communicate business findings, not to narrate every coding step. Format the report in APA style. A strong report usually follows this structure:

1. **Executive framing (short paragraph)** — state the business problem, the two companies, and why semantic embeddings matter in this module
2. **Q1: Model comparison** — report exact performance differences across TF-IDF, scratch Word2Vec, and pretrained GloVe
3. **Q2: Embedding geometry** — explain what the projections reveal about separation, overlap, and section-level structure
4. **Q3: Business interpretation** — connect semantic search behavior to analyst workflow, efficiency, and limitations
5. **Business recommendation** — recommend when embedding-based analysis is worth the cost and when simpler baselines are sufficient

For full credit, your report should:

- Use exact values copied from the executed notebook, not rounded guesses or rewritten claims
- Reference the most important tables and charts directly in the surrounding prose
- Interpret the evidence in business language, not only technical language
- Present figures and tables with readable titles, labels, and captions
- Avoid contradictions between the report and the notebook
- End with a clear recommendation, limitation, and next step

## Notebook Reproducibility Standard

`Fully executable on Colab` means a grader can run the notebook from top to bottom in a fresh runtime without repairing hidden state. Your notebook should:

- Mount Drive once and define paths once near the top
- Install and import packages in a single setup section
- Execute in linear order without duplicated blocks that redefine core variables inconsistently
- Save intermediate artifacts before later sections reuse them
- Include short markdown cells that explain what each major section is doing and how to interpret outputs
- Show diagnostic evidence such as embedding quality checks, projection plots, classifier metrics, and example nearest-neighbor results

Avoid notebooks that depend on manually rerunning cells out of order, local file paths, hidden variables from earlier experiments, or report claims that are not supported by executed outputs.

Suggested notebook structure:

1. Configuration, Drive mount, package installs
2. Load and inspect Assignment 1 corpus
3. Train Word2Vec skip-gram from scratch
4. Load pretrained GloVe vectors
5. Build and train PyTorch MLP classifiers (scratch vs pretrained)
6. UMAP / t-SNE projection and visualization
7. Yahoo Finance annual return quintile analysis
8. Q1–Q3 analysis and Business Recommendation

# Objective {.unnumbered}

:::{.callout-note}
Assignment 1 used frequency counts (TF-IDF) to distinguish two rival companies by vocabulary alone. Assignment 2 asks a deeper question: does **learned semantic meaning** — where words that appear in similar contexts become close in vector space — provide richer representations of these companies?
:::

:::{.callout-important}
**Core question:** Does encoding financial language into a semantic vector space improve company discrimination — and what does the geometry of the embedding space reveal about corporate rivalry?
:::

# Load Assignment 1 Corpus {#sec-load}


In [ ]:
from pathlib import Path
import json
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() or (candidate / ".git").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root()
A1_DIR = REPO_ROOT / "deliverables-solutions" / "Assignments" / "outputs" / "M01_A_sol"
A2_DIR = REPO_ROOT / "deliverables-solutions" / "Assignments" / "outputs" / "M02_A_sol"
A2_DIR.mkdir(parents=True, exist_ok=True)

# Google Drive version kept here as comments for future Colab use.
# from google.colab import drive
# drive.mount("/content/drive", force_remount=True)
# A1_DIR = Path("/content/drive/MyDrive/assignment01")
# A2_DIR = Path("/content/drive/MyDrive/assignment02")
# A2_DIR.mkdir(parents=True, exist_ok=True)

chunks_df = pd.read_csv(A1_DIR / "chunks.csv")
print(chunks_df.shape)
chunks_df.info()
chunks_df.describe(include="all")
chunks_df.head()

with open(A1_DIR / "vocab.json", encoding="utf-8") as f:
    a1_vocab = json.load(f)
print(f"Assignment 1 vocabulary size: {len(a1_vocab)}")

companies = chunks_df["company"].dropna().unique().tolist()
tickers = chunks_df[["company", "ticker"]].drop_duplicates().set_index("company")["ticker"].to_dict()
company_a, company_b = companies[:2]
ticker_a, ticker_b = tickers[company_a], tickers[company_b]
print("Companies:", company_a, "vs.", company_b)

# Keep the assignment scope to exactly two rival companies.
# If you changed M01's company list, rerun M01 and this cell will pick up that order.
chunks_df = chunks_df[chunks_df["company"].isin([company_a, company_b])].copy()
print(f"Filtered to two-company assignment scope: {len(chunks_df):,} chunks")


📊 **Required:** Stacked bar chart of chunk counts by company and by item type (Item 7 vs Item 7A vs Item 8).

# Train a Word2Vec Embedding from Scratch {#sec-word2vec}

## Tokenize and Train


In [ ]:
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

sentences = [simple_preprocess(text) for text in chunks_df["chunk_text"]]

w2v_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,       # skip-gram
    epochs=10,
    workers=4)

w2v_model.save(str(A2_DIR / "word2vec_scratch.bin"))
print(f"Vocabulary size: {len(w2v_model.wv)}")


## Inspect Nearest Neighbors for Finance Terms

For each of the five terms below, retrieve the top-10 nearest neighbors and build a DataFrame:


In [ ]:
finance_terms = ["liquidity", "leverage", "risk", "capital", "growth"]

rows = []
for term in finance_terms:
    if term in w2v_model.wv:
        for neighbor, sim in w2v_model.wv.most_similar(term, topn=10):
            rows.append({"seed_term": term, "neighbor": neighbor,
                         "similarity": round(sim, 4)})

neighbors_df = pd.DataFrame(rows)
print(neighbors_df.shape)
neighbors_df.info()
neighbors_df.describe()
neighbors_df.head(20)


📊 **Required:** For two of the five terms, show a horizontal bar chart of the top-10 nearest neighbors sorted by similarity. Label axes and include the term in the chart title.

## Per-Company Nearest Neighbors

Train separate Word2Vec models on Company A chunks only and Company B chunks only. For the term `"risk"`, compare the top-5 nearest neighbors from each model in a side-by-side table:


In [ ]:
nn_comparison = pd.DataFrame({
    f"{company_a} neighbors": top5_a,
    f"{company_b} neighbors": top5_b})
nn_comparison


# Load Pretrained GloVe Vectors {#sec-glove}

Download GloVe 6B 100-dimensional vectors and load into a gensim-compatible format.


In [ ]:
# Local-first GloVe location. Download/unzip the GloVe file here when running locally.
# In Colab, you may instead uncomment the two shell commands below and set GLOVE_DIR = Path("/content/glove").
# !wget -q http://nlp.stanford.edu/data/glove.6B.zip
# !unzip -q glove.6B.zip -d /content/glove/
from pathlib import Path
from gensim.models import KeyedVectors
from gensim.scripts.glove2word2vec import glove2word2vec

GLOVE_DIR = A2_DIR / "glove"
GLOVE_DIR.mkdir(parents=True, exist_ok=True)
GLOVE_TXT = GLOVE_DIR / "glove.6B.100d.txt"
GLOVE_GENSIM = GLOVE_DIR / "glove_gensim.txt"

# Google Colab alternative:
# GLOVE_DIR = Path("/content/glove")
# GLOVE_TXT = GLOVE_DIR / "glove.6B.100d.txt"
# GLOVE_GENSIM = GLOVE_DIR / "glove_gensim.txt"

if not GLOVE_TXT.exists():
    raise FileNotFoundError(
        f"Missing {GLOVE_TXT}. Download glove.6B.zip and unzip glove.6B.100d.txt into this folder."
    )

if not GLOVE_GENSIM.exists():
    glove2word2vec(str(GLOVE_TXT), str(GLOVE_GENSIM))

glove_model = KeyedVectors.load_word2vec_format(str(GLOVE_GENSIM))
print(f"GloVe vocabulary size: {len(glove_model)}")


# PyTorch MLP Classifier {#sec-mlp}

## Build Mean-Pooled Chunk Vectors

For each chunk, embed tokens using Word2Vec (or GloVe), then mean-pool to get a single 100-d vector:


In [ ]:
import numpy as np

def mean_pool_w2v(text, model):
    tokens = simple_preprocess(text)
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(100)

def mean_pool_glove(text, model):
    tokens = simple_preprocess(text)
    vecs = [model[t] for t in tokens if t in model]
    return np.mean(vecs, axis=0) if vecs else np.zeros(100)

X_w2v = np.array([mean_pool_w2v(t, w2v_model) for t in chunks_df["chunk_text"]])
X_glove = np.array([mean_pool_glove(t, glove_model) for t in chunks_df["chunk_text"]])
print(f"Word2Vec embedding matrix: {X_w2v.shape}")
print(f"GloVe embedding matrix: {X_glove.shape}")


Show `.describe()` on the embedding matrix:


In [ ]:
pd.DataFrame(X_glove).describe()


## Train a 2-Layer MLP


In [ ]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

class MLPClassifier(nn.Module):
    def __init__(self, input_dim=100, hidden_dim=64, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)


Train for 30 epochs with AdamW optimizer and cross-entropy loss. Plot the training loss curve for both models.

📊 **Required:** Training loss curve with Word2Vec model and GloVe model on the same chart.

Evaluate both models and build a 3-row accuracy comparison table including the Assignment 1 BoW baseline:


In [ ]:
comparison_df = pd.DataFrame({
    "Method": ["A1: TF-IDF BoW (Linear)", "A2: Word2Vec scratch MLP",
               "A2: GloVe pretrained MLP"],
    "Accuracy": [a1_acc, w2v_acc, glove_acc]})
print(comparison_df)


📊 **Required:** Horizontal bar chart of all three method accuracies.

Save GloVe MLP weights:


In [ ]:
torch.save(glove_mlp.state_dict(), A2_DIR / "glove_mlp_weights.pt")


# Embedding Projection with UMAP {#sec-umap}

Project all GloVe chunk embeddings to 2D using UMAP (or t-SNE if UMAP is unavailable):


In [ ]:
import umap  # !pip install umap-learn

reducer = umap.UMAP(n_components=2, random_state=42)
X_2d = reducer.fit_transform(X_glove)

proj_df = pd.DataFrame(X_2d, columns=["x", "y"])
proj_df["company"] = chunks_df["company"].values
proj_df["item"] = chunks_df["item"].values
proj_df["year"] = chunks_df["year"].values

print(proj_df.shape)
proj_df.info()
proj_df.describe()
proj_df.head()


📊 **Required Plot 1:** 2D scatter plot colored by company. Title: "Embedding Space by Company." Do the two rivals form separate clusters or overlap?

📊 **Required Plot 2:** 2D scatter plot colored by item type (Item 7 vs Item 7A vs Item 8). Title: "Embedding Space by 10-K Section." Which section separates more cleanly?

# Yahoo Finance Annual Return Quintile Analysis {#sec-returns}

Pull annual returns for both companies and assign each filing chunk to a return quintile based on its year:


In [ ]:
import yfinance as yf

def get_annual_returns(ticker: str, start: str = "2019-12-31",
                       end: str = "2024-12-31") -> pd.Series:
    hist = yf.Ticker(ticker).history(start=start, end=end, interval="1mo")
    hist["year"] = hist.index.year
    return hist.groupby("year")["Close"].last().pct_change().dropna()

returns_a = get_annual_returns(ticker_a)
returns_b = get_annual_returns(ticker_b)

returns_df = pd.DataFrame({
    f"{company_a} return": returns_a,
    f"{company_b} return": returns_b})
print(returns_df.shape)
returns_df.info()
returns_df.describe()
returns_df


Merge a `return_quintile` column (1=lowest, 5=highest annual return) onto `proj_df` by year and company.

📊 **Required Plot 3:** 2D scatter plot colored by `return_quintile`. Title: "Embedding Space by Return Quintile." Do high-return-year filings cluster differently from low-return-year filings?

# Q1: Does Moving from TF-IDF to Embeddings Improve Discrimination? {#sec-q1}

Report all three accuracy numbers in the comparison table:

| Method | Accuracy |
|--------|----------|
| A1: TF-IDF BoW Classifier | |
| A2: Word2Vec scratch MLP | |
| A2: GloVe pretrained MLP | |

📊 **Required:** Include the accuracy bar chart from Section 5 in your Word document.

**Answer in your Word document:** Does the switch from TF-IDF to learned embeddings improve company discrimination? Does using pretrained GloVe outperform scratch Word2Vec? What does the pattern suggest about whether these companies' 10-K language is already well-represented in general English corpora?


# Q2: What Does the Embedding Geometry Reveal? {#sec-q2}

Refer to your two UMAP scatter plots (by company and by item type).

**Answer in your Word document:** Do the two rivals cluster separately or overlap in the embedding space? Which 10-K section (Item 7, Item 7A, or Item 8) produces cleaner separation? Does this match the section-level accuracy result from Assignment 1? Explain the alignment or discrepancy in 2–3 sentences.


# Q3: What Does an Analyst Gain and Lose With Semantic Search? {#sec-q3}

## Business Recommendation — Analyst Efficiency

Your Word2Vec model trained on 5 years of 10-K filings has implicitly encoded each company's strategic vocabulary. If a financial analyst used nearest-neighbor queries in this embedding space instead of reading raw filings:

**Answer in your Word document (≤1 page):**

- What types of questions could they answer quickly using nearest-neighbor search?
- What would they miss that a careful reader would catch?
- Based on the average `char_count` of your chunks, estimate roughly how many pages of 10-K text a single nearest-neighbor lookup "replaces" in terms of reading time.
- Does the UMAP return-quintile plot suggest any relationship between how a company writes about risk and how it performs in the market? State what you see or do not see.


# Deliverables Summary {.unnumbered}

| Artifact | Location |
|----------|----------|
| `word2vec_scratch.bin` | `/assignment02/` on Drive |
| `glove_mlp_weights.pt` | `/assignment02/` on Drive |
| 3-method accuracy comparison table | In notebook + Word doc |
| UMAP scatter plots (3) | In notebook + Word doc |
| Nearest-neighbor comparison table | In notebook + Word doc |
| Annual return DataFrame | In notebook + Word doc |
| AI disclosure document | Blackboard submission |

# Use of Generative AI {.unnumbered}

You may use generative AI tools for code assistance, debugging, and explanation. You must:

- List every tool used
- State exactly which task each tool supported
- Include the complete prompt and the corresponding output excerpt or exported response, not only a one-line summary
- Include a shared conversation link when the tool supports sharing; if it does not, say so explicitly
- Identify which AI-generated outputs were used in the final submission and which were discarded or corrected
- Explain how you validated the output against notebook execution, saved artifacts, and manual inspection

When possible, preserve the original interaction using a share link or export feature. Examples include:

- **Gemini:** export the response to Google Docs or share the chat if available
- **ChatGPT:** submit a ChatGPT shared link
- **Claude:** submit a Claude shared chat link
- **Perplexity:** submit a shared thread link

If a share link is not available, include the full prompt and the relevant output in the disclosure document as screenshots or copied text.

Your AI disclosure should include the following fields:

1. Tool name and provider
2. Date used
3. Task supported
4. Complete prompt(s)
5. Share link(s), if available
6. Output excerpt(s) or exported response used in final submission
7. Validation steps you performed
8. Corrections you made after validation

Submit AI disclosure as a separate document. Generic statements such as "I used AI for help" are not sufficient for full credit.
